# Parent Atlas — Catalog, Motion & GPU Lab (v1)

**Runnable CPU baseline; optional GPU paths.** A stand-alone reference notebook for a **500+ concept taxonomy**, 256-bit semantic masks, provenance-aware asset catalog, 768→256 MRL retrieval contract, graph ranking, cluster IDs, deterministic motion FSM, an 8 MiB *encoded-asset* pack plan and JSON exports for your existing Parent Atlas SvelteKit/PG18/SeaweedFS pipeline.

**No private case uploads, no production DB calls, no automatic model weights, no GPU requirement.** All default examples are explicitly **synthetic**. It does **not** infer real identities, criminal intent or legal outcomes. It never marks evidence reviewed or triggers scene Save. Do **not** run the hosted Colab lane on confidential evidence; use a separately authorized local environment if needed.

Hardware: in Google Colab choose **Runtime → Change runtime type → GPU** if desired. Colab GPU type (including RTX PRO 6000 G4) is not guaranteed. WSL2/local workstations can execute the CPU and supported CUDA sections. The NVIDIA model projects have distinct, sometimes incompatible stacks: test **one** in a separate environment. All optional install flags default to `False`.

In [ ]:
from pathlib import Path
import os, sys, json, math, uuid, time, hashlib, platform, struct, subprocess, importlib.util
from dataclasses import dataclass, asdict
from collections import defaultdict
import numpy as np

OUT = Path('/content/parent_atlas_exports' if Path('/content').exists() else './parent_atlas_exports')
OUT.mkdir(parents=True, exist_ok=True)
MAX_PACK_BYTES = 8 * 1024 * 1024  # encoded assets policy; not GPU VRAM
TAXONOMY_VERSION = 'atlas-taxonomy-demo-v1'
RIG_VERSION = 'atlas-humanoid-v1'
SEED = 24
np.random.seed(SEED)
print({'python': sys.version.split()[0], 'platform': platform.platform(), 'out': str(OUT), 'budget_bytes': MAX_PACK_BYTES})
try:
    import torch
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
    print('Torch:', torch.__version__, 'device:', DEVICE)
    if DEVICE == 'cuda':
        print('GPU:', torch.cuda.get_device_name(0), 'VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory/2**30,1))
except ImportError:
    torch = None
    DEVICE = 'cpu'
    print('Torch not installed. CPU baseline still works.')

## 1. Identity policy and ontology imports

A **taxonomy concept** gets a stable versioned key, with UUIDv5 deterministically derived here for the demo. **Persisted scene entities** receive UUIDv7 from PostgreSQL 18. A **scene-local handle** is a `uint32` that may change at each pack build. A **blob** is SHA-256-addressed; a **vertex/UV/keyframe** only needs an array index. New revisions refer to immutable ancestors, rather than being assigned arbitrary new taxonomy meanings.

Set `ONTOLOGY_PATH` to a **non-sensitive, manually exported JSON or CSV** containing `key,label,domain` if you want to inspect your real 500+ taxonomy. Otherwise the notebook makes 530 clearly synthetic entries. This is not evidence ingestion.

In [ ]:
import csv
ONTOLOGY_PATH = ''  # optional public/non-sensitive local JSON or CSV file
BASE_CONCEPTS = [
 ('environment.beach','Beach','environment'), ('environment.city','City','environment'),
 ('building.house','House','building'), ('prop.bonfire','Bonfire','prop'),
 ('character.humanoid','Humanoid','character'), ('hair.card','Hair card','appearance'),
 ('motion.walk','Walk','motion'), ('motion.idle','Idle','motion'),
 ('material.brick','Brick','material'), ('evidence.video','Video evidence reference','evidence'),
 ('behavior.inspect','Inspect','behavior'), ('geometry.blockout','Blockout primitive','geometry')
]
def concept_id(key):
    return str(uuid.uuid5(uuid.NAMESPACE_URL, 'urn:parent-atlas:concept:'+key))
def load_concepts(path):
    if path:
        p=Path(path)
        if p.suffix.lower()=='.json':
            records=json.loads(p.read_text())
        elif p.suffix.lower()=='.csv':
            with p.open(newline='',encoding='utf-8') as f: records=list(csv.DictReader(f))
        else: raise ValueError('Only .json or .csv')
        rows=[(x['key'],x['label'],x['domain']) for x in records]
    else:
        rows=BASE_CONCEPTS + [('demo.synthetic.%03d'%i,'Synthetic concept %03d'%i,'synthetic') for i in range(530)]
    if len(set(k for k,_,_ in rows))!=len(rows): raise ValueError('Duplicate taxonomy keys')
    return [dict(key=k,label=label,domain=d,concept_id=concept_id(k),taxonomy_version=TAXONOMY_VERSION) for k,label,d in rows]
concepts=load_concepts(ONTOLOGY_PATH)
concept_by_key={c['key']:c for c in concepts}
print('concept_count',len(concepts),'synthetic_demo',not bool(ONTOLOGY_PATH))
assert len(concepts)>500

## 2. Linked tuples, graph and ranking

Represent semantic relations as `(subject_id, predicate, object_id, evidence_ref, provenance_kind)`. **Do not put unauthorized private evidence edges into exported packs.** Use Neo4j for authorized graph queries when available; NetworkX here is a small offline reference. Global PageRank is only a feature, never evidence of guilt or truth. Filter by case/access first, then lexical/embedding candidates, then graph expansion/rerank.

In [ ]:
import networkx as nx
RELATIONS = [
 ('building.house','contains','material.brick'),
 ('building.house','located_in','environment.city'),
 ('environment.beach','may_contain','prop.bonfire'),
 ('character.humanoid','can_perform','motion.walk'),
 ('character.humanoid','can_perform','motion.idle'),
 ('character.humanoid','may_have','hair.card'),
 ('character.humanoid','uses','behavior.inspect'),
 ('geometry.blockout','uses','material.brick')
]
triples=[dict(subject=concept_id(s),predicate=p,object=concept_id(o),source='synthetic_taxonomy',evidence_ref=None) for s,p,o in RELATIONS]
G=nx.DiGraph()
for c in concepts: G.add_node(c['concept_id'],label=c['label'],domain=c['domain'])
for s,p,o in RELATIONS: G.add_edge(concept_id(s),concept_id(o),relation=p)
pr=nx.pagerank(G,alpha=0.85)
print('top graph nodes',sorted([(G.nodes[k]['label'],round(v,4)) for k,v in pr.items()],key=lambda x:-x[1])[:6])

## 3. Compact masks and original synthetic embedding vectors

`256 bits = 32 bytes` is a feature filter, **not a unique ID** or 256-dimensional embedding. Bit collisions are possible. In production use the existing 500+ taxonomy keys and typed relation tables for exact filtering; reserve masks for quick *approximate/declared* features.

Default 768-dimensional vectors below are deterministic **synthetic fixtures**, **not EmbeddingGemma inference**. Real EmbeddingGemma 2 supports 768 and MRL truncation to 256/128/512, with renormalization. Every embedding row needs `model_id`, weights revision, modality and preprocessing revision.

In [ ]:
def bit256(flags):
    out=bytearray(32)
    for f in sorted(set(flags)):
        digest=hashlib.sha256(f.encode()).digest()
        bit=int.from_bytes(digest[:2],'little')%256
        out[bit//8] |= 1<<(bit%8)
    return bytes(out)
def mask_match_all(mask,required): return all((a&b)==b for a,b in zip(mask,required))
def synthetic_embedding(text,dim=768):
    # deterministic demonstration ONLY; not a model embedding
    seed=int.from_bytes(hashlib.sha256(text.encode()).digest()[:8],'little')
    v=np.random.default_rng(seed).standard_normal(dim).astype('float32')
    return v/(np.linalg.norm(v)+1e-12)
def mrl(v,d=256):
    assert 128<=d<=len(v)
    sub=np.asarray(v[:d],dtype=np.float32)
    return sub/(np.linalg.norm(sub)+1e-12)
assets=[
 dict(asset_key='synthetic.house.01',label='Four bedroom low-poly house',concept='building.house',scope='public-synthetic',format='GLB',rig=None,geometry_bytes=140_000,texture_bytes=75_000,motion_bytes=0),
 dict(asset_key='synthetic.beach.01',label='Beach terrain cell',concept='environment.beach',scope='public-synthetic',format='GLB',rig=None,geometry_bytes=110_000,texture_bytes=55_000,motion_bytes=0),
 dict(asset_key='synthetic.bonfire.01',label='Bonfire blockout',concept='prop.bonfire',scope='public-synthetic',format='GLB',rig=None,geometry_bytes=18_000,texture_bytes=18_000,motion_bytes=0),
 dict(asset_key='synthetic.person.01',label='Humanoid procedural character',concept='character.humanoid',scope='public-synthetic',format='GLB',rig=RIG_VERSION,geometry_bytes=44_000,texture_bytes=14_000,motion_bytes=12_000),
 dict(asset_key='synthetic.hair.01',label='Short hair-card set',concept='hair.card',scope='public-synthetic',format='GLB',rig=RIG_VERSION,geometry_bytes=9_000,texture_bytes=10_000,motion_bytes=0),
 dict(asset_key='synthetic.walk.01',label='Walk animation clip',concept='motion.walk',scope='public-synthetic',format='GLB',rig=RIG_VERSION,geometry_bytes=0,texture_bytes=0,motion_bytes=5_000),
]
for a in assets:
    a['asset_uuid']=str(uuid.uuid5(uuid.NAMESPACE_URL,'urn:parent-atlas:synthetic-asset:'+a['asset_key']))
    a['concept_uuid']=concept_by_key[a['concept']]['concept_id']
    a['features_hex']=bit256([a['concept'],a['format'].lower()]).hex()
    a['reference_sha256']=hashlib.sha256(('fixture:'+a['asset_key']).encode()).hexdigest() # fixture ID, NOT real GLB digest
    a['embedding_model']='DEMO_HASH_VECTOR_NOT_EMBEDDINGGEMMA'
V=np.stack([synthetic_embedding(a['label']) for a in assets]); q=synthetic_embedding('bonfire at a beach')
coarse=np.stack([mrl(v,256) for v in V]) @ mrl(q,256)
candidate_ids=np.argsort(-coarse)[:4]
full=V[candidate_ids] @ q
ranked=[(assets[int(candidate_ids[i])]['label'],round(float(full[i]),4)) for i in np.argsort(-full)]
print('synthetic ranked demo (not meaningful semantic search)',ranked)
assert len(bit256(['hair.card']))==32

## 4. Graph + cluster annotation (not asset identity)

For small datasets use CPU scikit-learn/NetworkX. For large batches, run optional cuML KMeans, cuGraph or UMAP in a **separate RAPIDS-compatible environment**. `KMeans` clusters and UMAP coordinates are versioned analysis *products*. Do not assign ontology meanings to unsupervised clusters automatically.

In [ ]:
from sklearn.cluster import KMeans
K=min(3,len(assets))
CLUSTER_RUN = str(uuid.uuid5(uuid.NAMESPACE_URL, 'urn:parent-atlas:demo-cluster:v1:k3:seed24'))
labels=KMeans(n_clusters=K,n_init=10,random_state=SEED).fit_predict(V)
cluster_assignments=[dict(asset_uuid=a['asset_uuid'],cluster_run_id=CLUSTER_RUN,cluster_label=int(label),embedding_model=a['embedding_model']) for a,label in zip(assets,labels)]
print('cluster assignments (synthetic vectors)', [(a['asset_key'],int(c)) for a,c in zip(assets,labels)])

## 5. GPU tensor awareness (optional, no RTX required)

PyTorch/CUDA speeds **numeric operations**, not hex conversion or automatic DAG construction. The only reason to move this batch to GPU is a measured speedup at useful scale. **Never treat cosine similarity as legal certainty.**

In [ ]:
if torch is not None:
    X=torch.from_numpy(V)
    query=torch.from_numpy(q)
    if DEVICE=='cuda': X=X.to('cuda');query=query.to('cuda')
    scores=(X @ query).detach().cpu().numpy()
    np.testing.assert_allclose(scores,V@q,rtol=1e-4,atol=1e-5)
    print('matrix validation passed on',DEVICE,'shape',tuple(X.shape))
else:
    print('No torch; numpy result',np.round(V@q,3).tolist())
print('Hex SHA is for integrity/display only, not a vector acceleration technique.')

## 6. Blockout DAG, motion/behavior and an 8 MiB pack plan

For runtime select a **small finite-state machine** or behavior tree for `Idle/Walk/Inspect/Sit`; no QLoRA model should run on each animation frame. Motion generators only emit candidates for review. An 8 MiB cap below governs **encoded assets/compiled scene packs**, not PyTorch VRAM, browser decoded RAM, or GPU texture allocations.

No new PASC binary writer is introduced here. Export the plan for the existing versioned PASC compiler and existing N64 tile-budget cache; avoid two format authorities. The generated scene is explicitly synthetic and preview-only.

In [ ]:
SCENE_ID=str(uuid.uuid5(uuid.NAMESPACE_URL,'urn:parent-atlas:scene:synthetic-beach-house-v1'))
recipe_nodes=[
 dict(node_id='beach',asset_key='synthetic.beach.01',depends_on=[]),
 dict(node_id='house',asset_key='synthetic.house.01',depends_on=['beach']),
 dict(node_id='bonfire',asset_key='synthetic.bonfire.01',depends_on=['beach']),
 dict(node_id='actor',asset_key='synthetic.person.01',depends_on=['beach']),
 dict(node_id='hair',asset_key='synthetic.hair.01',depends_on=['actor']),
 dict(node_id='walk',asset_key='synthetic.walk.01',depends_on=['actor']),
]
D=nx.DiGraph()
for n in recipe_nodes:
    D.add_node(n['node_id'])
    for dep in n['depends_on']: D.add_edge(dep,n['node_id'])
assert nx.is_directed_acyclic_graph(D)
order=list(nx.topological_sort(D)); assert len(order)==len(recipe_nodes)
asset_by_key={a['asset_key']:a for a in assets}
# dedup resource payload by immutable key for this demonstration
used={n['asset_key'] for n in recipe_nodes}
bytes_by_kind={k:sum(asset_by_key[key][k] for key in used) for k in ('geometry_bytes','texture_bytes','motion_bytes')}
record_plan_bytes=64 + len(recipe_nodes)*(64+64+32) # reference PASC layout, excludes dependency section etc.
encoded_budget=sum(bytes_by_kind.values())+record_plan_bytes
assert encoded_budget<=MAX_PACK_BYTES
# prototype FSM, not learned policy or a finding about any real person
TRANSITIONS={('idle','start_walk'):'walking',('walking','stop'):'idle',('idle','inspect'):'inspecting',('inspecting','finish'):'idle'}
def step_fsm(state,event): return TRANSITIONS.get((state,event),state)
assert step_fsm(step_fsm('idle','start_walk'),'stop')=='idle'
print({'dag_order':order,'encoded_bytes_estimate':encoded_budget,'budget_pct':round(encoded_budget/MAX_PACK_BYTES*100,2),'state_example':step_fsm('idle','inspect')})

## 7. Versioned, review-gated outputs and source/provenance metadata

Write only *synthetic* catalog and plan JSON into the notebook workspace. In production: metadata/revisions in PostgreSQL 18; GLB, motion, images, masks and derived frames as immutable SeaweedFS objects; graph projection in Neo4j; optional short-lived Valkey retrieval cache. **Never use a generated scene as original evidence.** Store `source_sha256`, evidence ID, source revision, timestamp interval, model/adapter version, transform and reviewer decision in a *separate* binding record. Retain multiple hypotheses without asserting guilt, intent, or identity.

In [ ]:
scene_manifest={
 'schema':'parent-atlas-scene-plan.v1','scene_id':SCENE_ID,'scope':'public-synthetic',
 'status':'DRAFT_SYNTHETIC_NOT_REVIEWED','taxonomy_version':TAXONOMY_VERSION,
 'source_evidence_refs':[], 'provenance':{'kind':'synthetic-demo','created_by':'offline_colab_reference'},
 'nodes':recipe_nodes,'assembly_order':order,
 'required_asset_keys':sorted(used), 'byte_accounting':{**bytes_by_kind,'pasc_reference_layout_bytes':record_plan_bytes,'sum_estimated_encoded_bytes':encoded_budget,'max_encoded_bytes':MAX_PACK_BYTES},
 'material_policy':'reuse_existing_n64_tile_budget_cache',
 'rig_policy':RIG_VERSION,
 'motion_bindings':[{'actor_node_id':'actor','clip_asset_key':'synthetic.walk.01','source_cue':None,'rig_version':RIG_VERSION,'review_required':True}],
 'save_approved':False,'publish_to_seaweedfs':False,
}
files={
 'synthetic_ontology.json':concepts,
 'synthetic_linked_tuples.json':triples,
 'synthetic_assets.json':assets,
 'synthetic_clusters.json':cluster_assignments,
 'synthetic_scene_plan.json':scene_manifest,
 'training_adapter_example.jsonl':None,
}
for filename,payload in files.items():
    if payload is not None:
        (OUT/filename).write_text(json.dumps(payload,indent=2,sort_keys=True),encoding='utf-8')
# Illustrative supervised structured-output example. NOT drawn from evidence. No training is performed.
example={'input':{'prompt':'Add a walking actor to a synthetic beach scene','available_assets':['synthetic.person.01','synthetic.walk.01']},'target':{'action':'propose_scene_draft','asset_key':'synthetic.person.01','motion_key':'synthetic.walk.01','requires_review':True}}
(OUT/'training_adapter_example.jsonl').write_text(json.dumps(example)+'\n',encoding='utf-8')
(OUT/'manifest_hashes.json').write_text(json.dumps({p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in OUT.iterdir() if p.is_file() and p.name!='manifest_hashes.json'},indent=2),encoding='utf-8')
print('exported',[(p.name,p.stat().st_size) for p in OUT.iterdir()])

## 8. Real EmbeddingGemma 2 (opt-in)

**Off by default**: This model may require access to the Hugging Face repository, dependencies and substantial RAM/VRAM. Use only public/synthetic input in cloud notebooks. Production embedding metadata MUST include model revision and normalization dimensions. Never mix original EmbeddingGemma and EmbeddingGemma 2 spaces or compare synthetic demo vectors to model vectors. Install from the [Google model guide](https://developers.googleblog.com/embeddinggemma-2-the-developer-guide/).

In [ ]:
ENABLE_REAL_EMBEDDINGS=False
if ENABLE_REAL_EMBEDDINGS:
    import sys, subprocess
    subprocess.check_call([sys.executable,'-m','pip','install','-q','sentence-transformers>=6.1.0'])
    from sentence_transformers import SentenceTransformer
    model=SentenceTransformer('google/embeddinggemma-2')
    real=model.encode([a['label'] for a in assets],normalize_embeddings=True,convert_to_numpy=True)
    assert real.shape[1]==768
    real_256=np.stack([mrl(v,256) for v in real]); print('real EmbeddingGemma2 shape',real.shape,real_256.shape)
else:
    print('Skipped external model download; verified synthetic MRL contract above.')

## 9. Optional NVIDIA RAPIDS clustering (separate install decision)

Colab includes some NVIDIA data science libraries on some runtimes, but check availability. If missing, use the [official Colab RAPIDS guide](https://docs.nvidia.com/datascience/deployment/26.10/platforms/colab/index.html) rather than blindly installing into an environment already occupied by motion models. Outputs are analysis-run metadata only.

In [ ]:
INSTALL_RAPIDS=False
if INSTALL_RAPIDS:
    assert DEVICE=='cuda','RAPIDS install requires compatible GPU/driver'
    subprocess.check_call(['git','clone','--depth','1','https://github.com/rapidsai/rapidsai-csp-utils.git',str(OUT/'rapidsai-csp-utils')])
    subprocess.check_call([sys.executable,str(OUT/'rapidsai-csp-utils/colab/pip-install.py')])
    print('Restart runtime if installer requests it; rerun notebook after restart.')
else:
    print('RAPIDS installation deferred; CPU KMeans reference already ran.')

## 10. MotionBricks / Kimodo / ARDY / HairDeck: one experimental stack per runtime

Verified official repos and roles:

- **[MotionBricks](https://github.com/NVlabs/motionbricks):** target poses + Smart Primitives; ONNX demos and TensorRT/Warp/Newton runtime; documented tested GPU RTX 4090, not guaranteed to work on RTX 3060 Ti.
- **[Kimodo](https://github.com/nv-tlabs/kimodo):** controllable **offline** diffusion motion authoring; outputs must be retargeted to Atlas rig.
- **[ARDY](https://github.com/nv-tlabs/ardy):** interactive motion generation; requires CUDA/PyTorch and potentially gated Meta Llama 3 text encoder.
- **[HairDeck](https://github.com/hyz317/HairDeck):** separate research hair-card generator, **not NVIDIA's official hair generator**; its install targets Python 3.12, torch 2.8, CUDA 12.8 and gated auxiliary weights. Prefer deterministic Blender hair cards first.

Read repository licenses AND checkpoint/data licenses separately. Do not concatenate all `pip install` commands into the same notebook runtime. Setting `CLONE_PROJECT` clones source for inspection. Heavy setup and checkpoints require an isolated environment as documented by that project.

In [ ]:
REPOS={
 'motionbricks':'https://github.com/NVlabs/motionbricks.git',
 'kimodo':'https://github.com/nv-tlabs/kimodo.git',
 'ardy':'https://github.com/nv-tlabs/ardy.git',
 'hairdeck':'https://github.com/hyz317/HairDeck.git',
}
CLONE_PROJECT='none' # none | motionbricks | kimodo | ardy | hairdeck
INSTALL_SELECTED_PROJECT=False  # intentional second gate; heavyweight installs are NOT automatic
if CLONE_PROJECT!='none':
    assert CLONE_PROJECT in REPOS
    root=OUT/'experiments'/CLONE_PROJECT
    root.parent.mkdir(parents=True,exist_ok=True)
    if not root.exists(): subprocess.check_call(['git','clone','--depth','1',REPOS[CLONE_PROJECT],str(root)])
    print('Cloned:',root)
    if INSTALL_SELECTED_PROJECT:
        assert DEVICE=='cuda','Enable a CUDA runtime; check project requirements'
        if CLONE_PROJECT=='motionbricks':
            print('Requires git-lfs and uv, then: git lfs pull && uv sync in repo; TensorRT plans are GPU-specific.')
        elif CLONE_PROJECT=='kimodo':
            print('New isolated Python environment: install compatible torch; then pip install -e . (see README).')
        elif CLONE_PROJECT=='ardy':
            print('New isolated Python 3.11 env + CUDA-compatible torch, cmake/build-essential, then pip install -e ".[all]"; gated text encoder.')
        elif CLONE_PROJECT=='hairdeck':
            print('New isolated Python 3.12 / torch 2.8 CUDA 12.8 env then README requirements; gated SAM3/DINOv3 weights.')
        print('Install intentionally not automated: separate runtime and checkpoint terms must be chosen first.')
else:
    print('No optional source repos cloned. For each, follow official project environment and licensing instructions.')

## 11. QLoRA / Unsloth adapter preparation (opt-in, separate runtime)

**Do not train a model to infer criminal intent, witness truthfulness, or legal guilt from scene animation.** Train narrow structured-output *suggestion* tasks only on licensed/public/synthetic examples; keep source references and human review. Your local 3060 Ti 8 GB may fit some small 4-bit adapters but not every model/context; check actual VRAM. BF16 and attention implementations depend on GPU. **Do not automatically install xformers**: try built-in PyTorch SDPA first and only benchmark other kernels.

This notebook writes one synthetic supervised example above. It does **not** claim a valid training corpus. Use [Unsloth requirements](https://unsloth.ai/docs/get-started/fine-tuning-for-beginners/unsloth-requirements) and choose a known compatible model and separate GPU runtime before installing.

In [ ]:
INSTALL_UNSLOTH=False
if INSTALL_UNSLOTH:
    assert DEVICE=='cuda','Choose GPU runtime and confirm PyTorch CUDA before install.'
    subprocess.check_call([sys.executable,'-m','pip','install','-U','unsloth'])
    print('Installed Unsloth package only. No training/weight download; next choose compatible model/adapter.')
else:
    print('QLoRA deferred. Data gate: independently reviewed synthetic/authorized examples, train/test split, schema validation, baseline.')

## 12. Validation and export

The generated JSON is for **offline review** before an agent merges into the existing PostgreSQL18/SeaweedFS/N64/Three.js systems. Nothing here talks to port 8095, gRPC, Valkey or a production case. After review: compile against the *existing* PASC contract, validate case-scoped authority, and test scene reload, scrubbing, mesh/texture reference counts, and evidence provenance in the browser.

In [ ]:
assert len(concepts)>500 and len(set(c['concept_id'] for c in concepts))==len(concepts)
assert len(triples)==len(RELATIONS) and all(x['source']=='synthetic_taxonomy' for x in triples)
assert all(len(bytes.fromhex(a['features_hex']))==32 for a in assets)
assert nx.is_directed_acyclic_graph(D)
assert scene_manifest['save_approved'] is False and not scene_manifest['source_evidence_refs']
assert encoded_budget<=MAX_PACK_BYTES
assert all(a['scope']=='public-synthetic' for a in assets)
print('PASS: identity, >500 concepts, mask width, DAG, privacy, budget and review gates')
print('Output:', OUT.resolve())
try:
    from google.colab import files
    # Uncomment to explicitly download reviewed synthetic outputs:
    # import shutil; shutil.make_archive('/content/parent_atlas_exports','zip',str(OUT)); files.download('/content/parent_atlas_exports.zip')
except ImportError:
    pass

## 13. Video-RAG 8s/20s FFmpeg Keyframe Analysis + Depth Anything V2 Small Motion Classifier & License Gate

Enforces commercial-safe Apache-2.0 / MIT / Public Domain dataset & model admission before any QLoRA merge into `gemma4-legal`, deduplicates FFmpeg keyframes via 64-bit perceptual `dHash`, and classifies 2.5D kinematics (`YOLOX` + `SAM 2` + `Depth Anything V2 Small` `relative_monocular` + `:8095` transcript speech) into coalesced `[startMs, endMs)` N64 FSM motion cues (`IDLE`, `WALK`, `INSPECT`, `REACH`, `DIALOGUE`) bound to `evidence:<uuid>`.

In [ ]:
# ============================================================================
# SECTION 13: Video-RAG 8s/20s FFmpeg Keyframe Analysis + Depth Anything V2 Small
# (relative_monocular, Apache-2.0) Motion Classifier & Commercial License Gate
# ============================================================================
import math
import struct
import hashlib
import uuid
import json
import numpy as np

ARTIFACT_LICENSE_REGISTRY_V1 = {
    "depth-anything/Depth-Anything-V2-Small-hf": {"license": "APACHE_2_0", "commercial_safe": True},
    "depth-anything/Depth-Anything-V2-Base-hf":  {"license": "CC_BY_NC_4_0", "commercial_safe": False},
    "yolox_nano.onnx":                           {"license": "APACHE_2_0", "commercial_safe": True},
    "facebook/sam2-hiera-tiny":                  {"license": "APACHE_2_0", "commercial_safe": True},
    "pile-of-law/pile-of-law":                   {"license": "PUBLIC_DOMAIN_CC0", "commercial_safe": True},
    "coastalcph/lex_glue":                       {"license": "CC_BY_4_0", "commercial_safe": True},
    "openai/gsm8k":                              {"license": "MIT", "commercial_safe": True},
    "parent-atlas.qlora-scene-example.v1":       {"license": "APACHE_2_0", "commercial_safe": True},
    "liuhaotian/LLaVA-Instruct-150K":            {"license": "CC_BY_NC_4_0", "commercial_safe": False},
    "HuggingFaceH4/CodeAlpaca_20K":              {"license": "OPENAI_TOS_NC", "commercial_safe": False},
    "Salesforce/xlam-function-calling-60k":      {"license": "CC_BY_NC_4_0", "commercial_safe": False},
    "NVlabs/motionbricks":                       {"license": "NVIDIA_RESEARCH_NC", "commercial_safe": False},
}

def validate_artifact_license_v1(artifact_id: str, mode: str = "production_merge") -> bool:
    info = ARTIFACT_LICENSE_REGISTRY_V1.get(artifact_id)
    if not info:
        return False
    return bool(info["commercial_safe"]) if mode == "production_merge" else True

def dhash_hamming64(hex_a: str, hex_b: str) -> int:
    return (int(hex_a, 16) ^ int(hex_b, 16)).bit_count()

def classify_video_clip_motion_v1(evidence_id: str, clip_window_s: int, depth_model_id: str, keyframes: list[dict]) -> dict:
    if depth_model_id != "depth-anything/Depth-Anything-V2-Small-hf":
        raise ValueError(f"UNAPPROVED_DEPTH_MODEL:{depth_model_id}")
    if clip_window_s not in (8, 20):
        raise ValueError("clip_window_s must be 8 or 20 (SCENE-DEC-A)")
    clock_id = f"evidence:{evidence_id}"
    max_ms = clip_window_s * 1000
    start_pts = keyframes[0]["pts_ms"]
    deduped, dropped = [], 0
    for i, kf in enumerate(keyframes):
        if kf["pts_ms"] - start_pts > max_ms:
            break
        if 0 < i < len(keyframes) - 1 and not kf.get("scene_cut", False) and deduped:
            if dhash_hamming64(deduped[-1]["dhash"], kf["dhash"]) <= 2:
                dropped += 1
                continue
        deduped.append(kf)

    cues = []
    for i in range(len(deduped) - 1):
        c0, c1 = deduped[i], deduped[i + 1]
        b0, b1 = c0["bbox"], c1["bbox"]
        lat = math.hypot(0.5 * (b1[0] + b1[2]) - 0.5 * (b0[0] + b0[2]), 0.5 * (b1[1] + b1[3]) - 0.5 * (b0[1] + b0[3]))
        d_delta = abs(c1["rel_depth"] - c0["rel_depth"])
        reach = c1["upper_mask_delta"]
        h0, h1 = b0[3] - b0[1], b1[3] - b1[1]
        crouch = (h0 - h1) / h0 if h0 > 0 else 0.0
        state = "IDLE"
        if lat >= 0.08 or d_delta >= 0.12:
            state = "WALK"
        elif reach >= 0.25:
            state = "REACH"
        elif crouch >= 0.15:
            state = "INSPECT"
        elif c1.get("speech", False):
            state = "DIALOGUE"
        if cues and cues[-1]["fsmState"] == state and cues[-1]["endMs"] == c0["pts_ms"]:
            cues[-1]["endMs"] = c1["pts_ms"]
        else:
            cues.append({
                "eventId": f"evt:motion_{evidence_id[:8]}_{len(cues)}",
                "clockId": clock_id,
                "startMs": c0["pts_ms"],
                "endMs": c1["pts_ms"],
                "fsmState": state,
                "reviewStatus": "UNREVIEWED_HYPOTHESIS"
            })
    return {
        "clockId": clock_id,
        "depthAuthority": "RELATIVE_MONOCULAR_ORDERING_ONLY",
        "dedupedFrames": len(deduped),
        "droppedDuplicates": dropped,
        "cues": cues
    }

sample_kfs = [
    {"pts_ms": 0,    "scene_cut": True,  "dhash": "1111111111111111", "bbox": [0.10, 0.2, 0.25, 0.8], "rel_depth": 0.30, "upper_mask_delta": 0.02, "speech": False},
    {"pts_ms": 500,  "scene_cut": False, "dhash": "1111111111111111", "bbox": [0.10, 0.2, 0.25, 0.8], "rel_depth": 0.30, "upper_mask_delta": 0.02, "speech": False},
    {"pts_ms": 2000, "scene_cut": False, "dhash": "2222222222222222", "bbox": [0.35, 0.2, 0.50, 0.8], "rel_depth": 0.50, "upper_mask_delta": 0.05, "speech": False},
    {"pts_ms": 4500, "scene_cut": False, "dhash": "4444444444444444", "bbox": [0.36, 0.2, 0.51, 0.8], "rel_depth": 0.51, "upper_mask_delta": 0.35, "speech": False},
    {"pts_ms": 8000, "scene_cut": False, "dhash": "8888888888888888", "bbox": [0.36, 0.2, 0.51, 0.8], "rel_depth": 0.51, "upper_mask_delta": 0.02, "speech": True},
]
video_motion = classify_video_clip_motion_v1("0f067235-4957-4783-82a5-ad93bd71daab", 8, "depth-anything/Depth-Anything-V2-Small-hf", sample_kfs)
print("Video-RAG 8s Clip Cues:", [(c["fsmState"], c["startMs"], c["endMs"]) for c in video_motion["cues"]])

## 14. Prompt-to-Animation Synthesizer + N64 / NES 4bpp Low-Nibble-First LOD Texture Streamer

Binds classified Video-RAG `[startMs, endMs)` FSM intervals to 7-bone skeletal quaternion clips and streams low-nibble-first 4bpp indexed-palette textures (`NearestFilter`, `generateMipmaps = False`) across discrete Morton cell LOD tiers (`retro` 9 cells / `balanced` 16 cells / `detailed` 25 cells).

In [ ]:
# ============================================================================
# SECTION 14: Prompt-to-Animation Synthesizer + N64 / NES 4bpp Low-Nibble-First
# Discrete LOD Texture Streamer (NearestFilter, generateMipmaps = False)
# ============================================================================
LOD_CELL_CAPS = {
    "retro":    {"cells": 9,  "cpu_mib": 8,  "gpu_mib": 32,  "upload_mib_per_frame": 1},
    "balanced": {"cells": 16, "cpu_mib": 16, "gpu_mib": 64,  "upload_mib_per_frame": 2},
    "detailed": {"cells": 25, "cpu_mib": 32, "gpu_mib": 128, "upload_mib_per_frame": 4},
}

def pack_4bpp_low_nibble_first(indices_u8: np.ndarray) -> bytes:
    flat = np.asarray(indices_u8, dtype=np.uint8).ravel()
    out = bytearray((len(flat) + 1) // 2)
    for i, val in enumerate(flat):
        nib = int(val) & 0x0F
        if (i & 1) == 0:
            out[i >> 1] |= nib
        else:
            out[i >> 1] |= (nib << 4)
    return bytes(out)

def prompt_and_videorag_to_n64_animation_stream(prompt: str, motion_res: dict, lod_tier: str = "balanced") -> dict:
    tile_16x16 = np.tile(np.arange(16, dtype=np.uint8), 16)
    packed_tile = pack_4bpp_low_nibble_first(tile_16x16)
    tile_sha = hashlib.sha256(packed_tile).hexdigest()
    bound_tracks = []
    for cue in motion_res["cues"]:
        bound_tracks.append({
            "eventId": cue["eventId"],
            "clockId": cue["clockId"],
            "intervalMs": [cue["startMs"], cue["endMs"]],
            "clipName": cue["fsmState"],
            "targetRig": "parent-atlas-7bone-v1",
            "reviewStatus": cue["reviewStatus"]
        })
    return {
        "prompt": prompt,
        "lodTier": lod_tier,
        "residencyPolicy": LOD_CELL_CAPS[lod_tier],
        "texture4Bpp": {
            "format": "atlas-indexed-palette16-v1",
            "width": 16,
            "height": 16,
            "generateMipmaps": False,
            "filter": "NearestFilter",
            "byteLength": len(packed_tile),
            "sha256": tile_sha,
            "packedHex": packed_tile.hex()
        },
        "animationBindings": bound_tracks
    }

anim_stream = prompt_and_videorag_to_n64_animation_stream("Studio apartment dinner discussion", video_motion, "balanced")
print(f"N64 Stream LOD={anim_stream['lodTier']} | 4bpp Tile={anim_stream['texture4Bpp']['byteLength']}B (generateMipmaps={anim_stream['texture4Bpp']['generateMipmaps']}) | Tracks={len(anim_stream['animationBindings'])}")

## 15. Parent Atlas Bitpacking-to-Hex + UUID + Kafka / RabbitMQ GPJSON Scene Envelope

Serializes 16-float ($64\text{-byte}$) world transform matrices, `ATLASBIT` 64-byte aligned LSB0 occupancy bitsets, 1-byte sparse routing tokens (`0x80` active gate | `0x7F` 7-bit AVX2 shifter), and 4bpp tile hashes into lowercase hex inside a deterministic **Kafka / RabbitMQ GPJSON (GeoJSON-compatible Local Uncalibrated Scene Envelope)**.

In [ ]:
# ============================================================================
# SECTION 15: Parent Atlas Bitpacking-to-Hex + UUID + Kafka / RabbitMQ GPJSON
# (GeoJSON-Compatible Local Uncalibrated Scene Envelope)
# ============================================================================
def create_atlasbit_64b_hex(bit_count: int, active_indices: list[int]) -> str:
    word_count = max(1, (bit_count + 63) // 64)
    rem = word_count % 8
    cap_words = max(8, word_count) if rem == 0 else word_count + (8 - rem)
    words = np.zeros(cap_words, dtype="<u8")
    for idx in active_indices:
        if 0 <= idx < bit_count:
            words[idx >> 6] |= np.uint64(1 << (idx & 63))
    digest = hashlib.sha256(words[:word_count].tobytes()).digest()
    header = struct.pack("<8sIIQ32s8s", b"ATLASBIT", 1, bit_count, word_count, digest, b"\x00" * 8)
    blob = header + words.tobytes()
    assert len(blob) % 64 == 0
    return blob.hex()

def build_kafka_gpjson_scene_envelope_v1(case_id: str, scene_id: str, revision: int, anim_packet: dict) -> dict:
    obj_uuid = str(uuid.uuid5(uuid.NAMESPACE_URL, f"urn:parent-atlas:{case_id}:{scene_id}:actor:0"))
    world_mat = np.eye(4, dtype="<f4").ravel()
    world_mat[12], world_mat[13], world_mat[14] = 1.25, 0.0, -2.5
    mat_hex = world_mat.tobytes().hex()
    atlasbit_hex = create_atlasbit_64b_hex(256, [0, 7, 64, 128])
    sparse_byte_hex = f"{(0x80 | 72):02x}"
    envelope = {
        "schema": "parent-atlas.kafka-gpjson-scene-envelope.v1",
        "kafkaTopic": "parent-atlas.scene.reconstruction.v1",
        "partitionKey": f"{case_id}:{scene_id}",
        "caseId": case_id,
        "sceneId": scene_id,
        "revision": revision,
        "canonicalAuthority": False,
        "coordinateFrame": "LOCAL_UNCALIBRATED",
        "gpjson": {
            "type": "FeatureCollection",
            "features": [
                {
                    "type": "Feature",
                    "id": obj_uuid,
                    "geometry": {"type": "Point", "coordinates": [1.25, 0.0, -2.5]},
                    "properties": {
                        "prefabId": "house.studio.v1",
                        "ontologyConceptId": "pa:actor.representation.neutral-humanoid",
                        "matrix4x4Hex128": mat_hex,
                        "atlasBitset64BHex": atlasbit_hex,
                        "sparseRoutingByteHex": sparse_byte_hex,
                        "texture4BppSha256": anim_packet["texture4Bpp"]["sha256"],
                        "animationBindings": anim_packet["animationBindings"]
                    }
                }
            ]
        }
    }
    canonical_bytes = json.dumps(envelope, sort_keys=True).encode("utf-8")
    envelope["envelopeSha256"] = hashlib.sha256(canonical_bytes).hexdigest()
    return envelope

kafka_gpjson = build_kafka_gpjson_scene_envelope_v1("11111111-2222-4333-8444-555555555555", "99999999-8888-4777-8666-555555555555", 1, anim_stream)
print(f"Kafka GPJSON Envelope Topic={kafka_gpjson['kafkaTopic']} | ObjectUUID={kafka_gpjson['gpjson']['features'][0]['id']} | MatrixHexLen={len(kafka_gpjson['gpjson']['features'][0]['properties']['matrix4x4Hex128'])} | SHA256={kafka_gpjson['envelopeSha256'][:16]}...")